# 2 — Provision Server & Setup Environment

This notebook:
1. Provisions the GPU server from the active lease.
2. Installs Docker + NVIDIA Container Toolkit.
3. Clones the project repo.
4. Prepares model files for all serving containers.

**Run after notebook 1 and after the lease is ACTIVE.**

In [ ]:
# runs in Chameleon Jupyter environment
from chi import server, context, lease, network
import os

context.version = "1.0"
context.choose_project()
context.choose_site(default="CHI@UC")

In [ ]:
# runs in Chameleon Jupyter environment
# CHANGE to match your Net ID from notebook 1
NET_ID     = "YOUR_NET_ID"
LEASE_NAME = f"serve_jellyfin_{NET_ID}"
REPO_URL   = "https://github.com/YOUR_ORG/YOUR_REPO.git"  # ← change to your repo

l = lease.get_lease(LEASE_NAME)
l.show()

In [ ]:
# runs in Chameleon Jupyter environment
username = os.getenv('USER')

s = server.Server(
    f"node-serve-jellyfin-{username}",
    reservation_id=l.node_reservations[0]["id"],
    image_name="CC-Ubuntu24.04-CUDA",
)
s.submit(idempotent=True)

In [ ]:
# runs in Chameleon Jupyter environment
s.associate_floating_ip()

In [ ]:
# runs in Chameleon Jupyter environment
s.refresh()
s.check_connectivity()

In [ ]:
# runs in Chameleon Jupyter environment
s.refresh()
s.show(type="widget")

## Open security groups

In [ ]:
# runs in Chameleon Jupyter environment
security_groups = [
    {"name": "allow-ssh",  "port": 22,   "description": "SSH"},
    {"name": "allow-8000", "port": 8000, "description": "FastAPI / serving endpoint"},
    {"name": "allow-8888", "port": 8888, "description": "Jupyter (optional)"},
]

for sg in security_groups:
    secgroup = network.SecurityGroup({
        "name": sg["name"],
        "description": sg["description"],
    })
    secgroup.add_rule(direction="ingress", protocol="tcp", port=sg["port"])
    secgroup.submit(idempotent=True)
    s.add_security_group(sg["name"])
    print(f"Added security group: {sg['name']}")

## Install Docker

In [ ]:
# runs in Chameleon Jupyter environment
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")

## Install NVIDIA Container Toolkit

In [ ]:
# runs in Chameleon Jupyter environment
s.execute(
    "curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey "
    "| sudo gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg "
    "&& curl -s -L https://nvidia.github.io/libnvidia-container/stable/deb/nvidia-container-toolkit.list "
    "| sed 's#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g' "
    "| sudo tee /etc/apt/sources.list.d/nvidia-container-toolkit.list"
)
s.execute("sudo apt-get update -qq")
s.execute("sudo apt-get install -y nvidia-container-toolkit")
s.execute("sudo nvidia-ctk runtime configure --runtime=docker")
# workaround for https://github.com/NVIDIA/nvidia-container-toolkit/issues/48
s.execute(
    "sudo jq 'if has(\"exec-opts\") then . "
    "else . + {\"exec-opts\": [\"native.cgroupdriver=cgroupfs\"]} end' "
    "/etc/docker/daemon.json | sudo tee /etc/docker/daemon.json.tmp "
    "&& sudo mv /etc/docker/daemon.json.tmp /etc/docker/daemon.json"
)
s.execute("sudo systemctl restart docker")
print("NVIDIA toolkit installed.")

In [ ]:
# runs in Chameleon Jupyter environment — verify GPU is visible inside Docker
stdout, rc = s.execute("docker run --rm --gpus all nvidia/cuda:12.1.0-base-ubuntu22.04 nvidia-smi")
print(stdout)

## Clone the project repository

In [ ]:
# runs in Chameleon Jupyter environment
stdout, rc = s.execute(f"git clone {REPO_URL} ~/jellyfin-recommender")
print(stdout)

## Prepare model files for all containers

The ONNX model lives under `serving/onnx/models/`.  
Copy it into `serving/multiworker/models/` so the multi-worker container can use it.

In [ ]:
# runs in Chameleon Jupyter environment
stdout, rc = s.execute(
    "cp -r ~/jellyfin-recommender/serving/onnx/models "
    "~/jellyfin-recommender/serving/multiworker/models"
)
print(stdout if stdout else "Done.")

In [ ]:
# runs in Chameleon Jupyter environment — confirm model files are in place
stdout, rc = s.execute("find ~/jellyfin-recommender/serving -name '*.pt' -o -name '*.onnx' | sort")
print(stdout)

## Install Python benchmark dependencies on the host

In [ ]:
# runs in Chameleon Jupyter environment
s.execute("sudo apt-get install -y python3-pip -qq")
s.execute("pip3 install --quiet requests numpy")
print("Benchmark dependencies installed.")

---
Server is ready.  
Note the floating IP printed in the widget above — you will need it in notebook 3.

**Next step:** open `3_run_experiments.ipynb`.